<a href="https://colab.research.google.com/github/theCyberLight/MyPortfolio/blob/main/LSTM%20Analysis%20of%20Nigerian%20Energy%20Generation%2C%20Resources...ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nigeria Energy Generation, Demand & Resources Analysis
## LSTM Deep Learning Forecasting — 2026 to 2030

> **Topic:** Analysis of Nigeria Energy Generation, Demand, and Resources for Effective Energy Transition (2016–2025)  
> **Tools:** TensorFlow/Keras · Pandas · NumPy · Matplotlib · Scikit-learn · OpenPyXL

---

### Expected Folder Structure
Place this notebook in the **same folder** as your `Peter_Database/` directory:

```
your_project/
├── Nigeria_Energy_LSTM.ipynb        ← this notebook
└── Peter_Database/
    ├── NESI Operational Data 2016.xlsx
    ├── NESI Operational Data 2017.xlsx
    ├── NESI Operational Data 2018.xlsx
    ├── NESI Operational Data 2019.xlsx
    ├── NESI Operational Data 2020.xlsx
    ├── NESI Operational Data_2021 - 2022.xlsx
    ├── NESI Operational Data 2023.xlsx
    ├── NESI Operational Data 2024.xlsx
    ├── NESI Operational Data 2025.xlsx
    ├── owid-energy-data.csv
    ├── electricity-prod-source-stacked (1).csv
    └── primary-energy-cons.csv
```

### Outputs
All results are saved to an `outputs/` folder created automatically:
- `Nigeria_Energy_Dataset_2016_2030.xlsx` — 5-sheet workbook
- `results_summary.json` — machine-readable results
- `fig1` through `fig7` — publication-quality PNG charts

---
## Cell 1 — Install Dependencies
Run this cell **once**. You can skip it on subsequent runs if packages are already installed.

In [ ]:
# Install all required packages
import sys
!{sys.executable} -m pip install tensorflow pandas numpy matplotlib scikit-learn openpyxl xlrd --quiet

---
## Cell 2 — Imports & Global Configuration

In [ ]:
import os
import json
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import pandas as pd
import numpy as np

# ── Jupyter-specific: render plots inline ──────────────────────────────────
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
tf.get_logger().setLevel('ERROR')
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print(f'TensorFlow  : {tf.__version__}')
print(f'Pandas      : {pd.__version__}')
print(f'NumPy       : {np.__version__}')
print(f'Matplotlib  : {matplotlib.__version__}')

# ── Paths ──────────────────────────────────────────────────────────────────
DB         = 'Peter_Database'   # source data folder
OUTPUT_DIR = 'outputs'          # all results written here
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── LSTM Hyperparameters ───────────────────────────────────────────────────
SEQ_LEN     = 3     # lookback window in years
FORECAST_N  = 5     # forecast horizon: 2026–2030
LSTM_EPOCHS = 400   # max epochs (early stopping applies)
LSTM_UNITS  = 64    # neurons in first LSTM layer

# ── Colour palette ─────────────────────────────────────────────────────────
COLORS = {
    'gas'      : '#E84855',
    'hydro'    : '#2196F3',
    'total'    : '#1B4332',
    'demand'   : '#FF6B35',
    'forecast' : '#9C27B0',
    'actual'   : '#2E86AB',
    'capacity' : '#F4A261',
    'renewable': '#52B788',
}

# ── Global plot style ──────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 10,
    'axes.titlesize'   : 12,
    'axes.labelsize'   : 10,
    'figure.dpi'       : 120,
    'savefig.dpi'      : 150,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.3,
})

print('\n✓  Configuration complete. Output folder:', OUTPUT_DIR)

TensorFlow  : 2.20.0
Pandas      : 2.2.2
NumPy       : 2.0.2
Matplotlib  : 3.10.0

✓  Configuration complete. Output folder: outputs


---
## Cell 3 — NESI Data Extraction
Parses all 10 NESI Operational Data Excel files (2016–2025).  
Each monthly sheet is read at the **plant level** — daily MWh/h values are summed across all power stations to get the daily grid total, then averaged to a monthly figure.

In [ ]:
def parse_nesi_sheet(filepath, sheet_name):
    """
    Parse one monthly sheet from a NESI Operational Data Excel file.

    Sheet layout (0-indexed rows):
        Row 0  : blank / file title
        Row 1  : 'AVERAGE GENERATION (MWH/H)' header
        Row 2  : date headers (one column per calendar day)
        Row 3  : column sub-headers
        Row 4+ : one row per power station; col 0 = plant name,
                 col 1+ = daily average generation in MWh/h

    Returns
    -------
    (month: int, avg_gen: float) or (None, None) if sheet is unreadable.
    """
    df = pd.read_excel(filepath, sheet_name=sheet_name, header=None)

    dates_row  = df.iloc[2, 1:]
    plant_data = df.iloc[4:, 1:].apply(pd.to_numeric, errors='coerce')

    # Daily grid total = sum across all plants
    daily_totals = plant_data.sum(axis=0).values
    dates        = pd.to_datetime(dates_row.values, errors='coerce')

    valid_pairs = [
        (d, v) for d, v in zip(dates, daily_totals)
        if pd.notna(d) and v > 0
    ]
    if not valid_pairs:
        return None, None

    ds, vs = zip(*valid_pairs)
    return pd.Timestamp(ds[0]).month, float(np.mean(vs))


def extract_nesi_annual(filepath, year_filter=None):
    """
    Extract all monthly averages from one NESI file.

    Parameters
    ----------
    filepath    : str
    year_filter : int or None  — pass an integer to skip sheets that do
                  not contain that year in their name (for combined files)

    Returns
    -------
    dict : {month_int: avg_gen_float}
    """
    xl      = pd.ExcelFile(filepath)
    monthly = {}
    for sheet in xl.sheet_names:
        if year_filter is not None and str(year_filter) not in sheet:
            continue
        try:
            month, avg = parse_nesi_sheet(filepath, sheet)
            if month is not None:
                monthly[month] = avg
        except Exception:
            pass
    return monthly


# Map every calendar year to its source file (and optional year filter)
NESI_FILES = {
    2016: ('NESI Operational Data 2016.xlsx',        None),
    2017: ('NESI Operational Data 2017.xlsx',        None),
    2018: ('NESI Operational Data 2018.xlsx',        None),
    2019: ('NESI Operational Data 2019.xlsx',        None),
    2020: ('NESI Operational Data 2020.xlsx',        None),
    2021: ('NESI Operational Data_2021 - 2022.xlsx', 2021),
    2022: ('NESI Operational Data_2021 - 2022.xlsx', 2022),
    2023: ('NESI Operational Data 2023.xlsx',        None),
    2024: ('NESI Operational Data 2024.xlsx',        None),
    2025: ('NESI Operational Data 2025.xlsx',        None),
}

nesi_monthly = {}
for yr, (fname, yf) in NESI_FILES.items():
    fpath = os.path.join(DB, fname)
    print(f'  Parsing {yr} → {fname}')
    nesi_monthly[yr] = extract_nesi_annual(fpath, year_filter=yf)

# Build tidy monthly DataFrame
monthly_rows = []
for yr in range(2016, 2026):
    for mo in range(1, 13):
        monthly_rows.append({
            'Year'        : yr,
            'Month'       : mo,
            'Avg_Gen_MWhh': nesi_monthly.get(yr, {}).get(mo, np.nan),
        })

monthly_df           = pd.DataFrame(monthly_rows)
monthly_df['Date']   = pd.to_datetime(monthly_df[['Year','Month']].assign(Day=1))
nesi_annual          = monthly_df.groupby('Year')['Avg_Gen_MWhh'].mean()

print('\nNESI Annual Average Generation (MWh/h):')
display(nesi_annual.rename('Avg_Gen_MWhh').to_frame().style.format('{:.2f}'))

  Parsing 2016 → NESI Operational Data 2016.xlsx


FileNotFoundError: [Errno 2] No such file or directory: 'Peter_Database/NESI Operational Data 2016.xlsx'

---
## Cell 4 — Load Supplementary Datasets (OWID, CSV Sources)

In [ ]:
owid      = pd.read_csv(os.path.join(DB, 'owid-energy-data.csv'))
elec_prod = pd.read_csv(os.path.join(DB, 'electricity-prod-source-stacked (1).csv'))
prim_cons = pd.read_csv(os.path.join(DB, 'primary-energy-cons.csv'))

ng_owid = owid[(owid['country'] == 'Nigeria') & owid['year'].between(2016, 2025)].reset_index(drop=True)
ng_prim = prim_cons[(prim_cons['Entity'] == 'Nigeria') & prim_cons['Year'].between(2016, 2025)].reset_index(drop=True)

print(f'OWID Nigeria rows   : {len(ng_owid)}')
print(f'Primary energy rows : {len(ng_prim)}')
print('\nOWID columns available:')
print([c for c in ng_owid.columns if any(k in c for k in
       ['electricity','energy','gas','hydro','demand','population','gdp','capita'])])

---
## Cell 5 — Build Unified Annual Dataset & Fill Data Gaps

**Unit conversion applied throughout:**  
`1 TWh/year = 1,000,000 MWh ÷ 8,760 hours = 114.16 MWh/h`

**Gap-filling strategy (per Chapter 3 methodology):**
1. Linear interpolation for gaps between two known values  
2. Forward-fill → backward-fill for leading/trailing edge gaps  
3. NESI gaps filled from OWID-converted MWh/h where NESI file is unavailable

In [ ]:
TWH_TO_MWHH = 1e6 / 8760   # 1 TWh/yr → MWh/h

def owid_val(col, yr):
    """Safely pull one value from the OWID Nigeria slice."""
    row = ng_owid[ng_owid['year'] == yr]
    if len(row) and col in row.columns:
        v = row[col].values[0]
        return float(v) if pd.notna(v) else np.nan
    return np.nan

def prim_val(yr):
    """Retrieve primary energy consumption (TWh) for a given year."""
    row = ng_prim[ng_prim['Year'] == yr]
    if len(row):
        v = row['primary_energy_consumption__twh'].values[0]
        return float(v) if pd.notna(v) else np.nan
    return np.nan

annual_rows = []
for yr in range(2016, 2026):
    gas_twh   = owid_val('gas_electricity',       yr)
    hydro_twh = owid_val('hydro_electricity',      yr)
    total_twh = owid_val('electricity_generation', yr)
    dem_twh   = owid_val('electricity_demand',     yr)
    ren_twh   = owid_val('renewables_electricity', yr)
    pc_kwh    = owid_val('per_capita_electricity', yr)
    pop       = owid_val('population',             yr)
    gdp       = owid_val('gdp',                   yr)
    prim_twh  = prim_val(yr)
    nesi_avg  = nesi_annual.get(yr, np.nan)

    annual_rows.append({
        'Year'                       : yr,
        # TWh series
        'Gas_Gen_TWh'                : gas_twh,
        'Hydro_Gen_TWh'              : hydro_twh,
        'Total_Gen_TWh'              : total_twh,
        'Demand_TWh'                 : dem_twh,
        'Renewables_Gen_TWh'         : ren_twh,
        'Primary_Energy_TWh'         : prim_twh,
        # Converted to MWh/h
        'Total_Gen_MWhh'             : total_twh * TWH_TO_MWHH if pd.notna(total_twh) else np.nan,
        'Gas_Gen_MWhh'               : gas_twh   * TWH_TO_MWHH if pd.notna(gas_twh)   else np.nan,
        'Hydro_Gen_MWhh'             : hydro_twh * TWH_TO_MWHH if pd.notna(hydro_twh) else np.nan,
        'Demand_MWhh'                : dem_twh   * TWH_TO_MWHH if pd.notna(dem_twh)   else np.nan,
        # NESI operational
        'NESI_Avg_Gen_MWhh'          : nesi_avg,
        'NESI_Annual_GWh'            : nesi_avg * 8760 / 1000  if pd.notna(nesi_avg)  else np.nan,
        # Socioeconomic
        'Per_Capita_Electricity_kWh' : pc_kwh,
        'Population_M'               : pop / 1e6 if pd.notna(pop) else np.nan,
        'GDP_B_USD'                  : gdp / 1e9 if pd.notna(gdp) else np.nan,
    })

annual_df = pd.DataFrame(annual_rows)

# ── Gap filling ─────────────────────────────────────────────────────────────
def fill_gaps(s):
    return s.interpolate(method='linear', limit_direction='both').ffill().bfill()

FILL_COLS = [
    'Gas_Gen_TWh','Hydro_Gen_TWh','Total_Gen_TWh','Demand_TWh',
    'Renewables_Gen_TWh','Per_Capita_Electricity_kWh','Population_M',
    'GDP_B_USD','Primary_Energy_TWh','Total_Gen_MWhh','Gas_Gen_MWhh',
    'Hydro_Gen_MWhh','Demand_MWhh',
]

annual_df_filled = annual_df.copy()
for col in FILL_COLS:
    annual_df_filled[col] = fill_gaps(annual_df_filled[col])

# Fill remaining NESI gaps from OWID-converted values
mask = annual_df_filled['NESI_Avg_Gen_MWhh'].isna()
annual_df_filled.loc[mask, 'NESI_Avg_Gen_MWhh'] = annual_df_filled.loc[mask, 'Total_Gen_MWhh']
annual_df_filled['NESI_Annual_GWh'] = annual_df_filled['NESI_Avg_Gen_MWhh'] * 8760 / 1000

print('Filled Annual Dataset — 2016 to 2025:')
display(annual_df_filled[[
    'Year','Total_Gen_TWh','Gas_Gen_TWh','Hydro_Gen_TWh',
    'Demand_TWh','NESI_Avg_Gen_MWhh','Primary_Energy_TWh'
]].set_index('Year').style.format('{:.2f}'))

---
## Cell 6 — LSTM Architecture & Helper Functions

**Architecture:** Input → LSTM(64, return_sequences=True) → Dropout(10%) → LSTM(32) → Dropout(10%) → Dense(1, linear)  
**Training:** Adam optimiser · MSE loss · Early stopping (patience = 30)

In [ ]:
def build_sequences(scaled_arr, seq_len):
    """
    Convert a 1-D scaled time series into (X, y) supervised learning pairs.
    X shape : (n_samples, seq_len)
    y shape : (n_samples,)
    """
    X, y = [], []
    for i in range(len(scaled_arr) - seq_len):
        X.append(scaled_arr[i : i + seq_len])
        y.append(scaled_arr[i + seq_len])
    return np.array(X), np.array(y)


def build_lstm_model(seq_len, units=64):
    """
    Stacked LSTM regression model.

    Layer 1 : LSTM(units, return_sequences=True)  — captures temporal patterns
    Layer 2 : Dropout(0.10)                        — regularisation
    Layer 3 : LSTM(units // 2)                     — compressed representation
    Layer 4 : Dropout(0.10)
    Layer 5 : Dense(1, linear)                     — regression output
    """
    model = Sequential([
        LSTM(units, return_sequences=True, input_shape=(seq_len, 1)),
        Dropout(0.10),
        LSTM(units // 2),
        Dropout(0.10),
        Dense(1),
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


def train_model(series_values):
    """
    Normalise → build sequences → train LSTM.

    Returns
    -------
    model  : trained Keras model
    scaler : fitted MinMaxScaler (required to invert predictions)
    """
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled = scaler.fit_transform(series_values.reshape(-1, 1)).flatten()

    X, y = build_sequences(scaled, SEQ_LEN)
    X    = X.reshape(X.shape[0], SEQ_LEN, 1)

    model      = build_lstm_model(SEQ_LEN, LSTM_UNITS)
    early_stop = EarlyStopping(monitor='loss', patience=30, restore_best_weights=True)
    model.fit(X, y, epochs=LSTM_EPOCHS, batch_size=4, verbose=0, callbacks=[early_stop])
    return model, scaler


def recursive_forecast(model, scaler, history_values, n_steps):
    """
    Multi-step recursive forecast.
    Each new prediction is appended to the input window and used
    as input for the next step — standard for multi-year energy forecasting.

    Returns np.ndarray of shape (n_steps,) in original units.
    """
    scaled_hist = list(scaler.transform(history_values.reshape(-1, 1)).flatten())
    predictions = []
    for _ in range(n_steps):
        window = np.array(scaled_hist[-SEQ_LEN:]).reshape(1, SEQ_LEN, 1)
        pred   = model.predict(window, verbose=0)[0, 0]
        scaled_hist.append(pred)
        predictions.append(pred)
    return scaler.inverse_transform(np.array(predictions).reshape(-1, 1)).flatten()


def compute_metrics(model, scaler, X_seq, y_seq):
    """Return RMSE, MAE, R² in original (inverse-transformed) units."""
    y_pred  = model.predict(X_seq, verbose=0)
    y_true  = scaler.inverse_transform(y_seq.reshape(-1, 1)).flatten()
    y_pred  = scaler.inverse_transform(y_pred).flatten()
    return (
        float(np.sqrt(mean_squared_error(y_true, y_pred))),
        float(mean_absolute_error(y_true, y_pred)),
        float(r2_score(y_true, y_pred)),
        y_true, y_pred
    )


print('✓  LSTM helper functions defined')
print(f'   SEQ_LEN={SEQ_LEN}  |  LSTM_UNITS={LSTM_UNITS}  |  MAX_EPOCHS={LSTM_EPOCHS}')

---
## Cell 7 — Train LSTM Models & Generate Forecasts (2026–2030)
⏳ This cell takes **3–8 minutes** depending on your machine. Progress is printed for each variable.

In [ ]:
# Variables to forecast
FORECAST_TARGETS = {
    'Total_Gen_TWh'     : 'Total Electricity Generation (TWh)',
    'Gas_Gen_TWh'       : 'Gas-Based Generation (TWh)',
    'Hydro_Gen_TWh'     : 'Hydropower Generation (TWh)',
    'Demand_TWh'        : 'Electricity Demand (TWh)',
    'Primary_Energy_TWh': 'Primary Energy Consumption (TWh)',
    'NESI_Avg_Gen_MWhh' : 'Average Generation (MWh/h)',
}

FORECAST_YEARS = list(range(2026, 2026 + FORECAST_N))
HIST_YEARS     = annual_df_filled['Year'].values

metrics_results = {}
forecasts       = {}

for col, label in FORECAST_TARGETS.items():
    print(f'\nTraining LSTM → {label}')

    # Prepare clean series
    raw = pd.Series(
        annual_df_filled[col].values.astype(float)
    ).interpolate().bfill().ffill().values

    # Train
    model, scaler = train_model(raw)

    # Evaluate on training sequences
    scaled     = scaler.transform(raw.reshape(-1, 1)).flatten()
    X_all, y_all = build_sequences(scaled, SEQ_LEN)
    X_all      = X_all.reshape(X_all.shape[0], SEQ_LEN, 1)
    rmse, mae, r2, _, _ = compute_metrics(model, scaler, X_all, y_all)
    print(f'  RMSE = {rmse:.3f}  |  MAE = {mae:.3f}  |  R² = {r2:.4f}')
    metrics_results[col] = {'Label': label, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

    # Recursive multi-step forecast 2026–2030
    forecasts[col] = recursive_forecast(model, scaler, raw, FORECAST_N)
    for yr, v in zip(FORECAST_YEARS, forecasts[col]):
        print(f'    {yr}: {v:.3f}')

print('\n' + '='*55)
print('  All LSTM models trained successfully!')
print('='*55)

---
## Cell 8 — Model Evaluation Metrics Summary

In [ ]:
metrics_df = pd.DataFrame(metrics_results).T[['Label','RMSE','MAE','R2']]
metrics_df.index.name = 'Variable'
metrics_df = metrics_df.reset_index()

def colour_r2(val):
    if   val >= 0.5 : return 'background-color: #d4edda; color: #155724'
    elif val >= 0.3 : return 'background-color: #fff3cd; color: #856404'
    else            : return 'background-color: #f8d7da; color: #721c24'

display(
    metrics_df.style
    .format({'RMSE': '{:.3f}', 'MAE': '{:.3f}', 'R2': '{:.4f}'})
    .applymap(colour_r2, subset=['R2'])
    .set_caption('LSTM Model Performance Metrics')
    .hide(axis='index')
)

---
## Cell 9 — Forecast Results Table (2026–2030)

In [ ]:
fc_rows = []
for i, yr in enumerate(FORECAST_YEARS):
    fc_rows.append({
        'Year'            : yr,
        'Total_Gen (TWh)' : round(float(forecasts['Total_Gen_TWh'][i]),    2),
        'Gas_Gen (TWh)'   : round(float(forecasts['Gas_Gen_TWh'][i]),      2),
        'Hydro_Gen (TWh)' : round(float(forecasts['Hydro_Gen_TWh'][i]),    2),
        'Demand (TWh)'    : round(float(forecasts['Demand_TWh'][i]),       2),
        'Prim_Energy(TWh)': round(float(forecasts['Primary_Energy_TWh'][i]),2),
        'Avg_Gen (MWh/h)' : round(float(forecasts['NESI_Avg_Gen_MWhh'][i]),1),
    })

fc_df = pd.DataFrame(fc_rows).set_index('Year')

display(
    fc_df.style
    .format('{:.2f}')
    .set_caption('LSTM Forecast Results — Nigeria Energy (2026–2030)')
    .background_gradient(cmap='Purples', axis=0)
)

---
## Cell 10 — Figure 1: Generation & Demand Forecast Dashboard (2×2)

In [ ]:
def shade_forecast(ax, start=2025.5, end=2031):
    ax.axvspan(start, end, alpha=0.08, color='purple')
    ax.axvline(x=2026, color='purple', linestyle='--', alpha=0.6, lw=1.2,
               label='Forecast Start (2026)')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(
    'Nigeria Electricity Generation Analysis (2016–2030)\n'
    'LSTM Forecasting commences from 2026',
    fontsize=14, fontweight='bold', y=1.01
)

# Panel A — Total generation
ax = axes[0, 0]
ax.plot(HIST_YEARS, annual_df_filled['Total_Gen_TWh'],
        'o-', color=COLORS['total'], lw=2.5, ms=6, label='Historical')
ax.plot(FORECAST_YEARS, forecasts['Total_Gen_TWh'],
        's--', color=COLORS['forecast'], lw=2.5, ms=8, label='LSTM Forecast')
shade_forecast(ax)
ax.set(title='Total Electricity Generation (TWh)', xlabel='Year', ylabel='TWh')
ax.legend(fontsize=8); ax.set_xlim(2015.5, 2031)

# Panel B — By source
ax = axes[0, 1]
ax.plot(HIST_YEARS, annual_df_filled['Gas_Gen_TWh'],
        'o-', color=COLORS['gas'],  lw=2.5, ms=6, label='Gas (Historical)')
ax.plot(HIST_YEARS, annual_df_filled['Hydro_Gen_TWh'],
        's-', color=COLORS['hydro'],lw=2.5, ms=6, label='Hydro (Historical)')
ax.plot(FORECAST_YEARS, forecasts['Gas_Gen_TWh'],
        '--', color=COLORS['gas'],  lw=2, alpha=0.8, label='Gas Forecast')
ax.plot(FORECAST_YEARS, forecasts['Hydro_Gen_TWh'],
        '--', color=COLORS['hydro'],lw=2, alpha=0.8, label='Hydro Forecast')
shade_forecast(ax)
ax.set(title='Generation by Source (TWh)', xlabel='Year', ylabel='TWh')
ax.legend(fontsize=8); ax.set_xlim(2015.5, 2031)

# Panel C — Generation vs Demand
ax = axes[1, 0]
ax.plot(HIST_YEARS, annual_df_filled['Total_Gen_TWh'],
        'o-', color=COLORS['total'],  lw=2.5, ms=6, label='Generation')
ax.plot(HIST_YEARS, annual_df_filled['Demand_TWh'],
        's-', color=COLORS['demand'], lw=2.5, ms=6, label='Demand')
ax.plot(FORECAST_YEARS, forecasts['Total_Gen_TWh'],
        '--', color=COLORS['total'],  lw=2, alpha=0.8, label='Gen Forecast')
ax.plot(FORECAST_YEARS, forecasts['Demand_TWh'],
        '--', color=COLORS['demand'], lw=2, alpha=0.8, label='Demand Forecast')
shade_forecast(ax)
ax.set(title='Generation vs Demand (TWh)', xlabel='Year', ylabel='TWh')
ax.legend(fontsize=8); ax.set_xlim(2015.5, 2031)

# Panel D — Primary energy
ax = axes[1, 1]
ax.plot(HIST_YEARS, annual_df_filled['Primary_Energy_TWh'],
        'o-', color='#2d6a4f', lw=2.5, ms=6, label='Historical')
ax.plot(FORECAST_YEARS, forecasts['Primary_Energy_TWh'],
        's--', color=COLORS['forecast'], lw=2.5, ms=8, label='LSTM Forecast')
shade_forecast(ax)
ax.set(title='Primary Energy Consumption (TWh)', xlabel='Year', ylabel='TWh')
ax.legend(fontsize=8); ax.set_xlim(2015.5, 2031)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig1_generation_forecast.png', bbox_inches='tight')
plt.show()
print('✓  fig1_generation_forecast.png saved')

---
## Cell 11 — Figure 2: NESI Average Generation (MWh/h) with Annotations

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

ax.fill_between(HIST_YEARS, annual_df_filled['NESI_Avg_Gen_MWhh'],
                alpha=0.3, color=COLORS['actual'])
ax.plot(HIST_YEARS, annual_df_filled['NESI_Avg_Gen_MWhh'],
        'o-', color=COLORS['actual'], lw=2.5, ms=7, label='Actual (MWh/h)')
ax.plot(FORECAST_YEARS, forecasts['NESI_Avg_Gen_MWhh'],
        's--', color=COLORS['forecast'], lw=2.5, ms=9, label='LSTM Forecast (MWh/h)')

# Annotate historical values
for yr, v in zip(HIST_YEARS, annual_df_filled['NESI_Avg_Gen_MWhh']):
    ax.annotate(f'{v:.0f}', (yr, v), xytext=(0, 8),
                textcoords='offset points', ha='center', fontsize=8)

# Annotate forecast values
for yr, v in zip(FORECAST_YEARS, forecasts['NESI_Avg_Gen_MWhh']):
    ax.annotate(f'{v:.0f}', (yr, v), xytext=(0, -16),
                textcoords='offset points', ha='center', fontsize=8,
                color=COLORS['forecast'])

shade_forecast(ax)
ax.set_title(
    'Nigeria Average Power Generation: 2016–2025 (Actual) & 2026–2030 (LSTM Forecast)\n'
    'Unit: MWh/h', fontsize=13, fontweight='bold'
)
ax.set(xlabel='Year', ylabel='Average Generation (MWh/h)')
ax.legend(fontsize=10); ax.set_xlim(2015.5, 2031)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig2_nesi_mwhh_forecast.png', bbox_inches='tight')
plt.show()
print('✓  fig2_nesi_mwhh_forecast.png saved')

---
## Cell 12 — Figure 3: Monthly Generation Heatmap

In [ ]:
pivot        = monthly_df.pivot(index='Year', columns='Month', values='Avg_Gen_MWhh')
month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(13, 6))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd', interpolation='nearest')
ax.set_xticks(range(12)); ax.set_xticklabels(month_labels)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
ax.set_title('Monthly Average Generation Heatmap (MWh/h) — 2016 to 2025',
             fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, label='Average Generation (MWh/h)')

max_val = np.nanmax(pivot.values)
for i in range(len(pivot.index)):
    for j in range(12):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=6.5,
                    color='black' if v < max_val * 0.7 else 'white')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig3_monthly_heatmap.png', bbox_inches='tight')
plt.show()
print('✓  fig3_monthly_heatmap.png saved')

---
## Cell 13 — Figure 4: Generation Mix Stacked Bar Chart

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
w      = 0.6
x_hist = np.arange(len(HIST_YEARS))
x_fc   = np.arange(len(HIST_YEARS), len(HIST_YEARS) + FORECAST_N)

gas_h   = annual_df_filled['Gas_Gen_TWh'].values
hydro_h = annual_df_filled['Hydro_Gen_TWh'].values
other_h = np.maximum(annual_df_filled['Total_Gen_TWh'].values - gas_h - hydro_h, 0)

gas_f   = forecasts['Gas_Gen_TWh']
hydro_f = forecasts['Hydro_Gen_TWh']
other_f = np.maximum(forecasts['Total_Gen_TWh'] - gas_f - hydro_f, 0)

# Historical — solid
ax.bar(x_hist, hydro_h, w, label='Hydropower',       color=COLORS['hydro'],     alpha=0.85)
ax.bar(x_hist, gas_h,   w, bottom=hydro_h,            label='Natural Gas',       color=COLORS['gas'],      alpha=0.85)
ax.bar(x_hist, other_h, w, bottom=hydro_h + gas_h,    label='Other/Renewable',   color=COLORS['renewable'],alpha=0.85)

# Forecast — hatched
ax.bar(x_fc, hydro_f, w,                           color=COLORS['hydro'],      alpha=0.45, hatch='//')
ax.bar(x_fc, gas_f,   w, bottom=hydro_f,            color=COLORS['gas'],       alpha=0.45, hatch='//')
ax.bar(x_fc, other_f, w, bottom=hydro_f + gas_f,    color=COLORS['renewable'], alpha=0.45, hatch='//')

ax.set_xticks(list(x_hist) + list(x_fc))
ax.set_xticklabels([str(y) for y in list(HIST_YEARS) + FORECAST_YEARS], rotation=45)
ax.axvline(x=len(HIST_YEARS) - 0.5, color='purple', linestyle='--', alpha=0.6)
ax.set_title('Nigeria Electricity Generation Mix (TWh)\n'
             'Actual 2016–2025  |  Forecast 2026–2030 (hatched)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Generation (TWh)'); ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig4_generation_mix.png', bbox_inches='tight')
plt.show()
print('✓  fig4_generation_mix.png saved')

---
## Cell 14 — Figure 5: Supply–Demand Gap (Shaded Area)

In [ ]:
all_years  = list(HIST_YEARS) + FORECAST_YEARS
gen_series = list(annual_df_filled['Total_Gen_TWh'].values) + list(forecasts['Total_Gen_TWh'])
dem_series = list(annual_df_filled['Demand_TWh'].values)    + list(forecasts['Demand_TWh'])

fig, ax = plt.subplots(figsize=(12, 6))

ax.fill_between(all_years, gen_series, dem_series,
                where=[d > g for g, d in zip(gen_series, dem_series)],
                alpha=0.3, color='red',   label='Demand Gap (Unmet)')
ax.fill_between(all_years, gen_series, dem_series,
                where=[g >= d for g, d in zip(gen_series, dem_series)],
                alpha=0.3, color='green', label='Surplus Generation')

ax.plot(all_years, gen_series, 'o-', color=COLORS['total'],  lw=2.5, ms=6, label='Generation')
ax.plot(all_years, dem_series, 's-', color=COLORS['demand'], lw=2.5, ms=6, label='Demand')
ax.axvline(x=2025.5, color='purple', linestyle='--', alpha=0.7, lw=1.5,
           label='Forecast Start (2026)')

ax.set_title('Nigeria Electricity: Generation vs Demand Gap (TWh)\n'
             '2016–2025 Historical  |  2026–2030 LSTM Forecast',
             fontsize=12, fontweight='bold')
ax.set(xlabel='Year', ylabel='TWh'); ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig5_demand_gap.png', bbox_inches='tight')
plt.show()
print('✓  fig5_demand_gap.png saved')

---
## Cell 15 — Figure 6: LSTM Model Performance (Fitted vs Actual)
⏳ Trains 6 lightweight models for visualisation — takes ~2 minutes.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('LSTM Model Performance — Fitted Values vs Actual Data',
             fontsize=13, fontweight='bold')

for idx, (col, label) in enumerate(FORECAST_TARGETS.items()):
    ax = axes[idx // 3, idx % 3]

    raw  = pd.Series(annual_df_filled[col].values.astype(float)).interpolate().bfill().ffill().values
    sc2  = MinMaxScaler()
    sc2d = sc2.fit_transform(raw.reshape(-1, 1)).flatten()
    X2, y2 = build_sequences(sc2d, SEQ_LEN)
    X2     = X2.reshape(X2.shape[0], SEQ_LEN, 1)

    m2 = build_lstm_model(SEQ_LEN, 64)
    m2.fit(X2, y2, epochs=200, batch_size=4, verbose=0,
           callbacks=[EarlyStopping(monitor='loss', patience=20, restore_best_weights=True)])

    y_pred2  = m2.predict(X2, verbose=0)
    y_true_i = sc2.inverse_transform(y2.reshape(-1, 1)).flatten()
    y_pred_i = sc2.inverse_transform(y_pred2).flatten()

    ax.plot(list(HIST_YEARS)[SEQ_LEN:], y_true_i, 'o-',
            color=COLORS['actual'],   lw=2, ms=5, label='Actual')
    ax.plot(list(HIST_YEARS)[SEQ_LEN:], y_pred_i, 's--',
            color=COLORS['forecast'], lw=2, ms=5, label='LSTM Fitted')

    m = metrics_results[col]
    ax.set_title(f'{label}\nR²={m["R2"]:.3f}  RMSE={m["RMSE"]:.2f}', fontsize=9)
    ax.set_xlabel('Year', fontsize=8); ax.tick_params(labelsize=8)
    ax.legend(fontsize=7)
    print(f'  {label[:35]:<35} done')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig6_model_performance.png', bbox_inches='tight')
plt.show()
print('✓  fig6_model_performance.png saved')

---
## Cell 16 — Figure 7: Per Capita Electricity Consumption

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(HIST_YEARS, annual_df_filled['Per_Capita_Electricity_kWh'],
       color=COLORS['actual'], alpha=0.8, edgecolor='white')

for yr, v in zip(HIST_YEARS, annual_df_filled['Per_Capita_Electricity_kWh']):
    ax.text(yr, v + 2, f'{v:.0f}', ha='center', va='bottom', fontsize=8)

ax.set_title('Nigeria Per Capita Electricity Consumption (kWh/person/year) — 2016–2025',
             fontsize=12, fontweight='bold')
ax.set(xlabel='Year', ylabel='kWh per Capita')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/fig7_per_capita.png', bbox_inches='tight')
plt.show()
print('✓  fig7_per_capita.png saved')

---
## Cell 17 — Export Excel Workbook (5 Sheets) & JSON Summary

In [ ]:
# ── Excel workbook ──────────────────────────────────────────────────────────
excel_path = f'{OUTPUT_DIR}/Nigeria_Energy_Dataset_2016_2030.xlsx'

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:

    # Sheet 1 — Historical annual summary (gap-filled)
    annual_df_filled.to_excel(writer, sheet_name='Annual_Summary_2016-2025', index=False)

    # Sheet 2 — LSTM forecasts
    fc_export = []
    for i, yr in enumerate(FORECAST_YEARS):
        row = {'Year': yr}
        for col in FORECAST_TARGETS:
            row[col] = round(float(forecasts[col][i]), 4)
        fc_export.append(row)
    pd.DataFrame(fc_export).to_excel(writer, sheet_name='LSTM_Forecast_2026-2030', index=False)

    # Sheet 3 — Combined 2016–2030
    hist_part = annual_df_filled.copy()
    hist_part['Data_Type'] = 'Historical'
    fc_part   = pd.DataFrame(fc_export)
    fc_part['Data_Type'] = 'LSTM Forecast'
    pd.concat([hist_part, fc_part], ignore_index=True).to_excel(
        writer, sheet_name='Combined_2016-2030', index=False
    )

    # Sheet 4 — Monthly NESI data
    monthly_df.to_excel(writer, sheet_name='Monthly_NESI_2016-2025', index=False)

    # Sheet 5 — Model metrics
    md = pd.DataFrame(metrics_results).T.reset_index()
    md.columns = ['Variable','Label','RMSE','MAE','R2']
    md.to_excel(writer, sheet_name='LSTM_Model_Metrics', index=False)

print(f'✓  Excel workbook saved → {excel_path}')

# ── JSON summary ─────────────────────────────────────────────────────────────
summary = {
    'annual_data'   : annual_df_filled[[
        'Year','Total_Gen_TWh','Gas_Gen_TWh','Hydro_Gen_TWh',
        'Demand_TWh','NESI_Avg_Gen_MWhh','Primary_Energy_TWh',
        'Per_Capita_Electricity_kWh','Population_M'
    ]].to_dict('records'),
    'forecasts'     : {col: [round(float(v), 4) for v in arr]
                       for col, arr in forecasts.items()},
    'forecast_years': FORECAST_YEARS,
    'metrics'       : {col: {k: (float(v) if k != 'Label' else v)
                             for k, v in m.items()}
                       for col, m in metrics_results.items()},
    'targets'       : FORECAST_TARGETS,
}
json_path = f'{OUTPUT_DIR}/results_summary.json'
with open(json_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'✓  JSON summary saved  → {json_path}')

---
## Cell 18 — Final Console Report

In [ ]:
print('=' * 65)
print('  LSTM FORECAST RESULTS — Nigeria Energy 2026 to 2030')
print('=' * 65)
print(f"  {'Year':<6} {'Gen(TWh)':<12} {'Gas(TWh)':<11} "
      f"{'Hydro(TWh)':<13} {'Demand(TWh)':<14} {'MWh/h'}")
print('  ' + '-' * 60)
for i, yr in enumerate(FORECAST_YEARS):
    print(f"  {yr:<6} "
          f"{forecasts['Total_Gen_TWh'][i]:<12.2f}"
          f"{forecasts['Gas_Gen_TWh'][i]:<11.2f}"
          f"{forecasts['Hydro_Gen_TWh'][i]:<13.2f}"
          f"{forecasts['Demand_TWh'][i]:<14.2f}"
          f"{forecasts['NESI_Avg_Gen_MWhh'][i]:.1f}")

print('\n  MODEL EVALUATION METRICS')
print('  ' + '-' * 60)
print(f"  {'Variable':<35} {'R²':>8} {'RMSE':>10} {'MAE':>10}")
print('  ' + '-' * 60)
for col, m in metrics_results.items():
    print(f"  {m['Label'][:35]:<35} {m['R2']:>8.4f} {m['RMSE']:>10.3f} {m['MAE']:>10.3f}")

print('\n  OUTPUT FILES')
print('  ' + '-' * 60)
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f:<45} {size/1024:>7.1f} KB")

print('\n  ✓  Analysis complete!')
print('=' * 65)